## Step 1: Import Libraries & Set API Key
Import LlamaIndex components for building a traditional vector-based RAG pipeline, and set the OpenAI API key for generating embeddings and LLM responses.

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
import os
os.environ["OPENAI_API_KEY"] = "your api key"


## Step 2: Define Benchmark Questions & Similarity Function
Create the answer key with 5 ground-truth questions and their expected answers from the budget PDF. Also define the similarity function that will grade the AI's responses by comparing them character-by-character.

In [ ]:
from difflib import SequenceMatcher
import pandas as pd

# Ground truth benchmark questions
benchmark = [
    {
        "question": "What is the estimated nominal GDP growth rate for 2025-26?",
        "expected": "The nominal GDP growth is estimated at 10.1%."
    },
    {
        "question": "What is the new rebate limit under the revised new income tax regime?",
        "expected": "Annual income of up to Rs 12 lakh will now receive a 100% rebate on taxable income."
    },
    {
        "question": "Which ministry received the highest budget allocation in 2025-26, and what is the amount?",
        "expected": "The Ministry of Defence received the highest allocation with Rs 6,81,210 crore."
    },
    {
        "question": "What is the proposed fiscal deficit target as a percentage of GDP for 2025-26?",
        "expected": "The fiscal deficit is targeted at 4.4% of GDP."
    },
    {
        "question": "What is the total allocation for the Pradhan Mantri Awas Yojana (Rural + Urban) in the new budget?",
        "expected": "It has been allocated Rs 78,126 crore."
    }
]

def similarity(a, b):
    return SequenceMatcher(None, a.lower(), b.lower()).ratio() * 100

## Step 3: Load Documents
Read and parse the budget PDF file using LlamaIndex's SimpleDirectoryReader. This converts the PDF into document objects that can be processed further.

In [ ]:

# 1. Load documents from a folder
documents = SimpleDirectoryReader(input_files=["./budget.pdf"]).load_data()

## Step 4: Build Vector Index
Chunk the loaded documents into smaller pieces, generate vector embeddings for each chunk using OpenAI, and store them in an in-memory vector index for similarity search.

In [ ]:
# 2. Build index (chunks + embeds automatically)
index = VectorStoreIndex.from_documents(documents)

## Step 5: Create Query Engine
Convert the vector index into a query engine that can accept natural language questions, retrieve relevant chunks via vector similarity, and generate answers using the LLM.

In [ ]:

# 3. Create query engine
query_engine = index.as_query_engine()

## Step 6: Ask a Sample Question
Test the query engine with a general question to confirm the RAG pipeline is working correctly end-to-end.

In [ ]:
# 4. Ask questions
response = query_engine.query("What is the main topic of these documents?")
print(response)

## Step 7: View Vector DB Contents
Inspect the internal vector store data and preview the first 100 characters of each stored document chunk to understand how the document was split.

In [ ]:
# 5. View vector DB contents
print(index.storage_context.vector_store._data)

for node_id, node in index.storage_context.docstore.docs.items():
    print(node_id, "→", node.text[:100])

## Step 8: Run Benchmark Evaluation & Calculate Accuracy
Ask all 5 benchmark questions one by one, compare each AI response to the expected answer using the similarity function, and calculate the overall General RAG accuracy score.

In [ ]:
general_results = []

# Question 1: What is the total budget allocation?
item1 = benchmark[0]
response1 = query_engine.query(item1["question"])
answer1 = str(response1)
score1 = similarity(answer1, item1["expected"])
general_results.append({
    "Question": item1["question"],
    "Expected": item1["expected"],
    "Predicted": answer1,
    "Score": score1
})

# Question 2: What is the allocation for education?
item2 = benchmark[1]
response2 = query_engine.query(item2["question"])
answer2 = str(response2)
score2 = similarity(answer2, item2["expected"])
general_results.append({
    "Question": item2["question"],
    "Expected": item2["expected"],
    "Predicted": answer2,
    "Score": score2
})

# Question 3: What is the allocation for healthcare?
item3 = benchmark[2]
response3 = query_engine.query(item3["question"])
answer3 = str(response3)
score3 = similarity(answer3, item3["expected"])
general_results.append({
    "Question": item3["question"],
    "Expected": item3["expected"],
    "Predicted": answer3,
    "Score": score3
})

# Question 4: Which sector received the highest budget allocation?
item4 = benchmark[3]
response4 = query_engine.query(item4["question"])
answer4 = str(response4)
score4 = similarity(answer4, item4["expected"])
general_results.append({
    "Question": item4["question"],
    "Expected": item4["expected"],
    "Predicted": answer4,
    "Score": score4
})

# Question 5: What are the key highlights of the budget?
item5 = benchmark[4]
response5 = query_engine.query(item5["question"])
answer5 = str(response5)
score5 = similarity(answer5, item5["expected"])
general_results.append({
    "Question": item5["question"],
    "Expected": item5["expected"],
    "Predicted": answer5,
    "Score": score5
})

general_df = pd.DataFrame(general_results)

general_accuracy = general_df["Score"].mean()

print("General RAG Accuracy:", round(general_accuracy, 2), "%")
general_df
